# Geospatial Time Series

Time series data often comes from physical locations — weather stations, power plants, grid nodes. TimeDataModel lets you attach geographic metadata directly to your series, keeping location and data together.

In [ ]:
from datetime import datetime, timedelta, timezone

import numpy as np

from timedatamodel import (
    TimeSeries,
    TimeSeriesTable,
    Frequency,
    DataType,
    GeoLocation,
)

base = datetime(2024, 1, 15, tzinfo=timezone.utc)
timestamps = [base + timedelta(hours=i) for i in range(24)]
rng = np.random.default_rng(42)

## GeoLocation — point coordinates

A `GeoLocation` is a simple `(latitude, longitude)` pair, validated on creation.

In [ ]:
oslo = GeoLocation(latitude=59.91, longitude=10.75)
print(f"Oslo: {oslo}")

bergen = GeoLocation(latitude=60.39, longitude=5.32)
print(f"Bergen: {bergen}")

In [ ]:
try:
    bad = GeoLocation(latitude=200, longitude=0)
except ValueError as e:
    print(f"Validation error: {e}")

## Attaching location to a TimeSeries

Pass a `GeoLocation` via the `location` parameter. It shows up in the repr.

In [ ]:
ts_oslo = TimeSeries(
    Frequency.PT1H,
    timezone="Europe/Oslo",
    timestamps=timestamps,
    values=(5.0 + rng.normal(0, 2, 24)).tolist(),
    name="temperature",
    unit="°C",
    description="Hourly temperature at Oslo Blindern",
    data_type=DataType.MEASUREMENT,
    location=oslo,
)
ts_oslo

## Per-column locations in a TimeSeriesTable

When columns represent different physical locations, attach one `GeoLocation` per column.

In [ ]:
tromsoe = GeoLocation(latitude=69.65, longitude=18.96)

table = TimeSeriesTable(
    Frequency.PT1H,
    timezone="Europe/Oslo",
    timestamps=timestamps,
    values=np.column_stack([
        5.0 + rng.normal(0, 2, 24),
        2.0 + rng.normal(0, 3, 24),
        -8.0 + rng.normal(0, 4, 24),
    ]),
    names=["Oslo", "Bergen", "Tromsø"],
    units=["°C", "°C", "°C"],
    locations=[oslo, bergen, tromsoe],
    data_types=[DataType.MEASUREMENT],
)
table

## GeoArea — polygon regions

`GeoArea` wraps a shapely `Polygon` for representing bidding zones, countries, or catchment areas. It requires the optional `shapely` dependency.

```bash
pip install timedatamodel[geo]
```

In [ ]:
from timedatamodel import GeoArea

try:
    no1 = GeoArea.from_coordinates(
        [
            (58.0, 6.0),
            (58.0, 12.0),
            (62.0, 12.0),
            (62.0, 6.0),
            (58.0, 6.0),
        ],
        name="NO1",
    )

    print(f"Area name:  {no1.name}")
    print(f"Bounds:     {no1.bounds}")
    print(f"Centroid:   {no1.centroid}")

    ts_area = TimeSeries(
        Frequency.PT1H,
        timestamps=timestamps,
        values=(120 + rng.normal(0, 15, 24)).tolist(),
        name="wind_NO1",
        unit="MW",
        location=no1,
    )
    ts_area
except ImportError:
    print("shapely not installed — skip this cell or: pip install timedatamodel[geo]")

## Custom attributes

The `attributes` dict lets you attach arbitrary key-value metadata — source system, fuel type, model version, etc.

In [ ]:
ts_rich = TimeSeries(
    Frequency.PT1H,
    timezone="Europe/Oslo",
    timestamps=timestamps,
    values=(80 + rng.normal(0, 10, 24)).tolist(),
    name="wind_farm_alpha",
    unit="MW",
    description="Measured output from Wind Farm Alpha",
    data_type=DataType.MEASUREMENT,
    location=oslo,
    attributes={
        "source": "SCADA",
        "fuel": "wind",
        "capacity_mw": "120",
        "operator": "NorthWind Energy",
    },
)

print(f"Attributes: {ts_rich.attributes}")
print(f"Capacity:   {ts_rich.attributes['capacity_mw']} MW")

## Frequency and DataType enums

TimeDataModel uses `StrEnum` types so values are both readable and machine-friendly.

In [ ]:
from timedatamodel import Frequency, DataType, TimeSeriesType

print("Available frequencies:")
for f in Frequency:
    td = f.to_timedelta()
    td_str = str(td) if td else "calendar-based" if f.is_calendar_based else "-"
    print(f"  {f.value:6s}  {td_str}")

print("\nData types:")
for dt in DataType:
    print(f"  {dt.value}")

print("\nTimeSeries types:")
for tst in TimeSeriesType:
    print(f"  {tst.value}")

## Metadata in JSON round-trips

Metadata like `data_type`, `attributes`, and `timeseries_type` survives JSON serialization.

In [ ]:
json_str = ts_rich.to_json()
ts_back = TimeSeries.from_json(json_str, location=oslo)

print(f"Name:        {ts_back.name}")
print(f"Data type:   {ts_back.data_type}")
print(f"Attributes:  {ts_back.attributes}")
print(f"Location:    {ts_back.location}")

## Summary

- `GeoLocation` attaches point coordinates (validated lat/lon)
- `GeoArea` wraps a shapely `Polygon` for regions (optional dependency)
- `TimeSeriesTable` supports per-column locations
- `attributes` dict stores arbitrary key-value metadata
- `Frequency`, `DataType`, and `TimeSeriesType` are `StrEnum` types for clean serialization
- All metadata round-trips through JSON